# Job: Ingestão para a Bronze

1. **Leitura das planilhas do XLSX da Landing para DataFrames:**
   - Informática e comunicação
   - Governança
   - Dicionário

   **Observação:** A planilha Dicionário contém as definições das variáveis (glossário de negócio) para os nomes de colunas codificados (ex.: MTIC001) usados nas demais planilhas.

2. **Limpeza e padronização do Dicionário:**
   - Trata células vazias e o preenchimento estrutural herdado do Excel de origem.

3. **Persistência das tabelas no schema Bronze usando Spark:**
   - munic_raw_data
   - munic_raw_dictionary

In [0]:
!pip install openpyxl

In [0]:
# Reinicia o kernel Python para usar o openpyxl
dbutils.library.restartPython()

In [0]:
%run ../environment/env_variables

In [0]:
%run ../environment/env_functions

In [0]:
import pandas as pd
import openpyxl
import requests
import re  # regex (expressões regulares)

In [0]:
# Variáveis de metadados
dataset_name = "IBGE_MUNIC"
# reference_year vem do env_variables -> não redefinir aqui

In [0]:
# Caminhos onde as tabelas finais serão armazenadas
# Colocar o nome do ano no nome da tabela (verificar se faremos isso)
bronze_tbl_raw_dict = f'{catalog_name}.{bronze_schema}.munic_raw_dict'
bronze_tbl_raw_info = f'{catalog_name}.{bronze_schema}.munic_raw_info_data'
bronze_tbl_raw_gov = f'{catalog_name}.{bronze_schema}.munic_raw_gov_data'


## 1. **Leitura das planilhas do XLSX da Landing para DataFrames:**

Workbook (wb) = o arquivo completo

Worksheet (ws) = as abas individuais dentro do arquivo

In [0]:
# Transforma cada planilha em pandas.DataFrame

# Dicionário das variáveis presentes nas planilhas de dados
df_dict = pd.read_excel(munic_file, sheet_name="Dicionário", header=None)

# Dados de Informática e Comunicação dos municípios
df_info_ws = pd.read_excel(
    munic_file,
    sheet_name="Informática e comunicação"
)

# Dados de Governança dos municípios
df_gov_ws = pd.read_excel(
    munic_file,
    sheet_name="Governanca"
)


In [0]:
df_dict.head()

#### 2. **Limpeza e padronização do DataFrame do Dicionário:**

In [0]:
def obtain_filled_cells(row):
    # Captura os valores preenchidos de uma linha da planilha de dicionário, eliminando as células vazias
    filled_cells = [str(val).strip() for val in row if pd.notna(val) and str(val).strip() != '']

    return filled_cells

In [0]:
def extract_var_code_meaning(row, var_pattern_list):
  """
    Extrai os códigos das variáveis e suas respectivas descrições da pesquisa
    a partir de uma linha bruta do dicionário MUNIC do IBGE.

    Parameters:
        row (pd.Series): Uma linha da planilha de dicionário ainda não formatada.
        var_pattern_list (list): Os prefixos temáticos de variáveis a casar (ex.: 'MTIC', 'MGOV', 'MREH').

    Returns:
        dict: {'cod_variable': <str>, 'meaning': <str>} se um código compatível com o padrão for encontrado,
              caso contrário, None.
    """

  # Usado para armazenar os padrões encontrados na linha via regex
  code_matches = list()
  filled_cells = obtain_filled_cells(row)

  # Verifica se a linha possui valores ou não
  if not filled_cells:
      return None

  # Cria as expressões regulares
  regex_expr_list = [rf'^{pattern}\d+' for pattern in var_pattern_list]


  # Verifica se alguma célula contém algum dos padrões de código e a adiciona à lista code_matches
  for re_expr in regex_expr_list:
      for val in filled_cells:
        if re.match(re_expr, val, flags=re.IGNORECASE):
           code_matches.append(val)


  # Se há algum valor com algum dos padrões de código
  if code_matches:
      cod_var = code_matches[0]

      # Localiza a descrição (o significado) do código
      desc_cells = [val for val in filled_cells if val != cod_var]

      # Se houver um único valor, usa-o; se houver mais, concatena-os; se não houver, retorna None para o código
      if desc_cells:
        if len(desc_cells) == 1:
          code_meaning = desc_cells[0]
        else:
          code_meaning = " - ".join(desc_cells) if desc_cells else None

      return {"cod_variable": cod_var, "meaning": code_meaning}

  # Caso contrário, retorna None
  return None

In [0]:
code_patterns = ['MTIC', 'MGOV']

# Obtém o significado dos códigos na planilha "Dicionário" (Ws)
# O resultado é uma lista de dicts (cada dict é um código e o seu significado)
meanings_raw_dict = [res for res in df_dict.apply(extract_var_code_meaning, axis=1, args=(code_patterns,)) if res is not None]

In [0]:
# Transforma em DataFrame
df_raw_dict = pd.DataFrame(meanings_raw_dict)

In [0]:
df_raw_dict.head()

## 3. **Persistência das tabelas no schema Bronze usando Spark:**

In [0]:
df_gov_ws.dtypes

In [0]:
df_info_ws.dtypes

In [0]:
def prepare_data_to_delta_format(df):
    """
    Prepara um pandas DataFrame para ingestão no Delta Lake:
    - Padroniza os nomes das colunas.
    - Converte todas as colunas do tipo object para o dtype 'string' do pandas.

    Parameters:
        df (pd.DataFrame): O pandas DataFrame de entrada.

    Returns:
        pd.DataFrame: O pandas DataFrame preparado
    """

    # Padroniza os nomes das colunas do pandas
    df.columns = [
        col.strip()
        .replace(" ", "_")
        .replace(";", "_")
        .replace(",", "_")
        .replace("-", "_")
        .lower()
        for col in df.columns
    ]

    # Converte todas as colunas do tipo object para o dtype 'string' do pandas
    # Isso é feito para o Spark não lançar erro de inferência de Object na passagem do pandas DataFrame para o Spark DataFrame
    for col in df.select_dtypes(include='object').columns:
         df[col] = df[col].astype('string')

In [0]:
prepare_data_to_delta_format(df_gov_ws)
prepare_data_to_delta_format(df_info_ws)

In [0]:
df_gov_ws.head()

### Processamento no Spark e gravação das tabelas bronze

In [0]:
spark_df_gov = spark.createDataFrame(df_gov_ws)
spark_df_info = spark.createDataFrame(df_info_ws)

In [0]:
spark_df_raw_dict = spark.createDataFrame(df_raw_dict)

#### Adição das colunas de metadados

In [0]:
# Auxilia a adicionar o ingestion_timestamp e o source_file
from pyspark.sql import functions as F

def add_ingestion_metadata(df, dataset_name, source_file, reference_year, source_url):
    return (
        df.withColumn("source_file", F.lit(source_file))
          .withColumn("source_url", F.lit(source_url))
          .withColumn("dataset_name", F.lit(dataset_name))
          .withColumn("reference_year", F.lit(reference_year))
          .withColumn("ingestion_timestamp", F.current_timestamp())
    )

In [0]:
# Variáveis definidas no início do dataset
spark_df_raw_info_governed = add_ingestion_metadata(
        spark_df_info, 
        dataset_name=dataset_name, 
        source_file=munic_file, 
        reference_year=reference_year,
        source_url=munic_source_url
    )

spark_df_raw_gov_governed = add_ingestion_metadata(
        spark_df_gov, 
        dataset_name=dataset_name, 
        source_file=munic_file, 
        reference_year=reference_year,
        source_url=munic_source_url
    )

In [0]:
spark_df_raw_dict_governed = add_ingestion_metadata(
        spark_df_raw_dict, 
        dataset_name=dataset_name, 
        source_file=munic_file, 
        reference_year=reference_year,
        source_url=munic_source_url
    )

In [0]:
display(
    spark_df_raw_gov_governed
        .select("source_file",
                 "ingestion_timestamp",
                 "dataset_name",
                 "reference_year",
                 "source_file")
        .show(5)
)

#### Gravação das tabelas no schema bronze

In [0]:
# Grava na bronze usando MERGE (upsert), particionada por ano_edicao
# O MERGE evita duplicatas: linhas correspondentes são atualizadas, novas linhas são inseridas.
# A evolução manual de schema (ALTER TABLE) permite que colunas de edições diferentes da pesquisa
# (ex.: MTIC015 em 2024 vs. MTIC016 em 2025) coexistam na mesma tabela.
# upsert_delta_table é importado do notebook env_functions

ano_edicao = int(reference_year) if isinstance(reference_year, (int, float, str)) else 2024

# Tabelas de dados — chave do MERGE: ano_edicao + cod_munic (PK do município)
upsert_delta_table(
    spark_df_raw_info_governed,
    bronze_tbl_raw_info,
    ["ano_edicao", "cod_munic"],
    ano_edicao,
    f"Ingestão de dados brutos da MUNIC {ano_edicao} — planilha Informática e Comunicação"
)

upsert_delta_table(
    spark_df_raw_gov_governed,
    bronze_tbl_raw_gov,
    ["ano_edicao", "cod_munic"],
    ano_edicao,
    f"Ingestão de dados brutos da MUNIC {ano_edicao} — planilha Governança"
)

In [0]:
# Tabela de dicionário — chave do MERGE: ano_edicao + cod_variable (PK da variável)
# Cada edição recebe sua própria versão do dicionário, então códigos com significados
# diferentes entre os anos coexistem com segurança.
upsert_delta_table(
    spark_df_raw_dict_governed,
    bronze_tbl_raw_dict,
    ["ano_edicao", "cod_variable"],
    ano_edicao,
    f"Dicionário de variáveis ingerido — pesquisa MUNIC {ano_edicao}"
)